# Problem: Write a Custom ReLU Operator in C++ (Step 1 — CPU Forward)

**Difficulty**: 🔴 Hard

**Companies**: NVIDIA, Meta, Google, xAI

---

### Problem Statement

PyTorch ships hundreds of operators — but when you need something custom (a fused kernel, a novel activation, a proprietary algorithm), you write a **C++ extension**.

In Step 1, you'll implement a **CPU-only ReLU forward pass** in C++ and register it with PyTorch's dispatcher so you can call it from Python just like `torch.relu`.

ReLU is simple by design: $$ \text{ReLU}(x) = \max(0, x) $$ — so you focus on the **mechanics of the extension system**, not the math.

---

### What you'll learn

| Concept | What it is |
|---|---|
| **`TORCH_LIBRARY`** | Macro that registers your operator's **schema** (name + type signature) with PyTorch's dispatcher |
| **Dispatcher** | PyTorch's runtime system that routes `my_op(tensor)` → the correct kernel for that tensor's device/dtype |
| **`TORCH_LIBRARY_IMPL`** | Macro that binds a **dispatch key** (e.g. `CPU`) to your C++ kernel implementation |
| **Tensor API** | `torch::Tensor`, `data_ptr<T>()`, `where()`, `zeros_like()`, `numel()` — the C++ surface for manipulating tensors |
| **`PYBIND11_MODULE`** | Required entry point so `cpp_extension.load()` can import the compiled `.so` |

---

### Requirements

1. **C++ ReLU kernel** — Implement `relu_cpu_forward` in `relu_op.cpp`. Strategy A (`torch::where`) or Strategy B (manual loop).
2. **Schema registration** — Register the operator name and signature using `TORCH_LIBRARY`.
3. **CPU binding** — Bind your kernel to the `CPU` dispatch key using `TORCH_LIBRARY_IMPL`.
4. **Python entry point** — Uncomment the `PYBIND11_MODULE` block so the `.so` can be imported.
5. **Build & validate** — Compile with `torch.utils.cpp_extension.load` and verify against `F.relu`.

---

### Constraints

- ✅ CPU only (CUDA comes in Step 2)
- ✅ Must work with `float32` tensors of any shape
- ❌ Do **not** call `torch.relu` or `F.relu` inside your C++ code

---

### The file to edit: `relu_op.cpp`

There are **4 blocks** to fill in. Each is a TODO comment — uncomment and complete the code.

```
Step 1: relu_cpu_forward()      ← the actual ReLU math
Step 2: TORCH_LIBRARY(... )     ← register schema
Step 3: TORCH_LIBRARY_IMPL(...) ← bind CPU kernel
Step 4: PYBIND11_MODULE(...)    ← Python entry point (just uncomment)
```

---

<details>
  <summary>💡 Hint 1: How TORCH_LIBRARY and the dispatcher fit together</summary>

  ```
  Python:  torch.ops.custom_relu.relu(x)
              │
              ▼
  ┌─────────────────────────────────────┐
  │         Dispatcher                   │
  │  Looks up "custom_relu::relu"       │
  │  Reads the schema (TORCH_LIBRARY)   │
  │  Finds the right kernel for "CPU"   │
  │  (from TORCH_LIBRARY_IMPL)          │
  └─────────────────────────────────────┘
              │
              ▼
  C++:     relu_cpu_forward(x)  ← your code!
  ```

  `TORCH_LIBRARY` is the **menu** (what ops exist).
  `TORCH_LIBRARY_IMPL` is the **kitchen** (who cooks them).
  `PYBIND11_MODULE` is the **door** (how Python gets in).
</details>

<details>
  <summary>💡 Hint 2: Tensor API cheat sheet</summary>

  ```cpp
  // Create tensors
  auto zeros = torch::zeros_like(input);
  auto out   = torch::empty_like(input);

  // Element-wise ops
  auto result = torch::where(condition, a, b);  // condition ? a : b

  // Iterate manually
  float* ptr = tensor.data_ptr<float>();
  for (int64_t i = 0; i < tensor.numel(); ++i) {
      ptr[i] = ...;
  }

  // Inspect shape
  auto shape = tensor.sizes();   // IntArrayRef
  int64_t n  = tensor.numel();   // total elements
  ```
</details>

<details>
  <summary>💡 Hint 3: The build call</summary>

  ```python
  from torch.utils.cpp_extension import load
  custom_relu = load(
      name="custom_relu",
      sources=["relu_op.cpp"],
      verbose=True,
  )
  ```

  This compiles your `.cpp` into a shared library and imports it — `TORCH_LIBRARY` runs at load time.
</details>

<details>
  <summary>💡 Hint 4: Calling your op from Python</summary>

  ```python
  # After load(), your op is registered under the namespace you chose:
  result = torch.ops.custom_relu.relu(x)
  ```
</details>

---


In [1]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load
import os

print(f"PyTorch version: {torch.__version__}")
print(f"Working directory: {os.getcwd()}")

CPP_FILE = "relu_op.cpp"
print(f"C++ source exists: {os.path.exists(CPP_FILE)}")


PyTorch version: 2.12.0+cu130
Working directory: /home/zireael/TorchLeet/torch/hard/custom-cpp-op
C++ source exists: True


### Step 1: Write the C++ kernel

Open `relu_op.cpp` in your editor and fill in the four TODO blocks:

1. **`relu_cpu_forward()`** — the ReLU kernel (Strategy A or B)
2. **`TORCH_LIBRARY`** — the operator schema
3. **`TORCH_LIBRARY_IMPL`** — bind to CPU dispatch key
4. **`PYBIND11_MODULE`** — just uncomment this block

The file lives at `torch/hard/custom-cpp-op/relu_op.cpp` relative to the repo root.

Below is what the skeleton looks like (read-only). Edit the actual file, not this cell.


### Step 2: Build the extension

Once you've filled in the TODOs, run this cell to compile your C++ code into a loadable PyTorch extension.

`torch.utils.cpp_extension.load` will:
1. Invoke the C++ compiler (g++/clang++)
2. Link against PyTorch's C++ libraries (libtorch)
3. Load the resulting `.so` into the Python process
4. `TORCH_LIBRARY` runs at import time, registering `custom_relu::relu` with the dispatcher


In [2]:
# Build and load the C++ extension
# This will fail until you complete the TODOs in relu_op.cpp
custom_relu = load(
    name="custom_relu",
    sources=["relu_op.cpp"],
    verbose=True,
)

print("\nExtension loaded successfully!")
print("Operator: torch.ops.custom_relu.relu")


[1/2] c++ -MMD -MF relu_op.o.d -DTORCH_EXTENSION_NAME=custom_relu -DTORCH_API_INCLUDE_EXTENSION_H -isystem /home/zireael/TorchLeet/.venv/lib/python3.13/site-packages/torch/include -isystem /home/zireael/TorchLeet/.venv/lib/python3.13/site-packages/torch/include/torch/csrc/api/include -isystem /home/zireael/.local/share/uv/python/cpython-3.13.9-linux-x86_64-gnu/include/python3.13 -fPIC -std=c++20 -c /home/zireael/TorchLeet/torch/hard/custom-cpp-op/relu_op.cpp -o relu_op.o 
[2/2] c++ relu_op.o -shared -L/home/zireael/TorchLeet/.venv/lib/python3.13/site-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o custom_relu.so

Extension loaded successfully!
Operator: torch.ops.custom_relu.relu


### Step 3: Test your custom op

Run the validation cells below. Your `custom_relu.relu` must match `torch.nn.functional.relu` on all test inputs.


In [3]:
# Sanity check: does the op exist?
try:
    op = torch.ops.custom_relu.relu
    print("Operator found:", op)
    print("Registered schemas:", op._schemas)
except AttributeError as e:
    print("Operator NOT found! Did the build succeed?")
    print("Check: TORCH_LIBRARY namespace should be 'custom_relu' and op name 'relu'")
    raise


Operator found: custom_relu.relu
Registered schemas: {'': custom_relu::relu(Tensor input) -> Tensor}


In [4]:
# Test 1: Positive values → unchanged
x_pos = torch.tensor([1.0, 2.0, 3.5, 100.0])
out = torch.ops.custom_relu.relu(x_pos)
expected = F.relu(x_pos)
print("Positive values:", out)
assert torch.allclose(out, expected), "Positive values test FAILED!"
print("PASSED")


Positive values: tensor([  1.0000,   2.0000,   3.5000, 100.0000])
PASSED


In [5]:
# Test 2: Negative values → zero
x_neg = torch.tensor([-1.0, -2.0, -0.5, -100.0])
out = torch.ops.custom_relu.relu(x_neg)
expected = F.relu(x_neg)
print("Negative values:", out)
assert torch.allclose(out, expected), "Negative values test FAILED!"
print("PASSED")


Negative values: tensor([0., 0., 0., 0.])
PASSED


In [6]:
# Test 3: Mixed values
x_mixed = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
out = torch.ops.custom_relu.relu(x_mixed)
expected = F.relu(x_mixed)
print("Mixed values:", out)
assert torch.allclose(out, expected), "Mixed values test FAILED!"
print("PASSED")


Mixed values: tensor([0., 0., 0., 1., 3.])
PASSED


In [7]:
# Test 4: Zero handling (ReLU(0) = 0)
x_zero = torch.zeros(5)
out = torch.ops.custom_relu.relu(x_zero)
expected = F.relu(x_zero)
print("Zeros:", out)
assert torch.allclose(out, expected), "Zero test FAILED!"
print("PASSED")


Zeros: tensor([0., 0., 0., 0., 0.])
PASSED


In [8]:
# Test 5: Multi-dimensional tensor
torch.manual_seed(42)
x_2d = torch.randn(4, 8)  # (batch, features)
out = torch.ops.custom_relu.relu(x_2d)
expected = F.relu(x_2d)
print("2D input shape:", x_2d.shape)
print("Max absolute error:", (out - expected).abs().max().item())
assert torch.allclose(out, expected), "2D tensor test FAILED!"
print("PASSED")


2D input shape: torch.Size([4, 8])
Max absolute error: 0.0
PASSED


In [9]:
# Test 6: Gradient check (does autograd flow through our custom op?)
# PyTorch automatically generates a backward for custom ops
# registered with TORCH_LIBRARY, since we didn't override Autograd.
# (PyTorch >= 2.x may show a deprecation warning — the gradient
#  is still correct; adding an AutogradCPU fallthrough silences it.)
x_grad = torch.randn(3, 4, requires_grad=True)
out = torch.ops.custom_relu.relu(x_grad)
loss = out.sum()
loss.backward()

print("Input grad:", x_grad.grad)
expected_grad = (x_grad > 0).float()  # ReLU gradient: 1 if x>0 else 0
print("Expected grad:", expected_grad)
assert torch.allclose(x_grad.grad, expected_grad), "Gradient test FAILED!"
print("PASSED — autograd flows through the custom op!")


Input grad: tensor([[1., 1., 0., 0.],
        [1., 1., 0., 1.],
        [1., 1., 1., 0.]])
Expected grad: tensor([[1., 1., 0., 0.],
        [1., 1., 0., 1.],
        [1., 1., 1., 0.]])
PASSED — autograd flows through the custom op!


### 🎉 All tests passed?

If every test above passes, you've successfully:
- Written a C++ kernel using PyTorch's Tensor API
- Registered it with `TORCH_LIBRARY`
- Bound it to the `CPU` dispatch key with `TORCH_LIBRARY_IMPL`
- Provided the Python entry point via `PYBIND11_MODULE`
- Built and loaded it from Python
- Verified correctness and autograd integration

**On to Step 2**: CUDA kernel + manual backward pass. The dispatcher makes this straightforward — just add `TORCH_LIBRARY_IMPL(custom_relu, CUDA, m)` and `TORCH_LIBRARY_IMPL(custom_relu, AutogradCPU, m)` blocks.
